# Flight Delays 2015 - BigQuery analysis (CCBD exam)

Self-documenting notebook for the *Cloud Computing and Big Data* exam (Universita di
Catania, AA 2025-2026). It drives **Google BigQuery** over the USDOT *2015 Flight Delays
and Cancellations* dataset (~5.3M flights, October dropped) and narrates the findings.

**The six questions**
1. On-time performance by airline
2. Delay propagation through the day
3. Bottleneck airports
4. Seasonality of delays and cancellations
5. Decomposition of delay causes
6. Distance vs in-flight recovery

Cloud setup, bucket and data loading live in `README.md`. Run top-to-bottom (*Run All*),
from the project root so that `sql/` and the credentials env var resolve.

## 0. Environment

This notebook runs in a dedicated venv registered as the **CCBD** Jupyter kernel. The
cell below pins dependencies with the `%pip` magic - **not** `!pip`, which can install
into a different interpreter than the kernel and cause `ImportError`. `%pip` installs
into the *kernel's* environment, keeping the notebook self-contained ("open -> Run All").

> Versions: resolve once on your machine, then pin exactly and `pip freeze > requirements.txt`
> as a lock file (see README). The pins below are a reasonable starting point.

In [ ]:
%pip install -q pandas==2.2.2 matplotlib==3.9.2 google-cloud-bigquery==3.25.0 db-dtypes==1.2.0 pyarrow==17.0.0

## 1. Authentication

The notebook authenticates to BigQuery with a **service-account key (JSON)**. The key
path is read from the `GOOGLE_APPLICATION_CREDENTIALS` environment variable - **never
hardcoded, never printed**. The key carries least-privilege roles (BigQuery Job User +
Data Editor; Storage Object Admin on the bucket only). See `README.md` section 1.1.

In [ ]:
import os
from google.cloud import bigquery
from google.oauth2 import service_account

PROJECT_ID = 'ccbd-20260603-gpappa'   # not secret - only the KEY stays out of the notebook
DATASET    = 'flights_2015'

key_path = os.environ['GOOGLE_APPLICATION_CREDENTIALS']   # path only; contents never printed
credentials = service_account.Credentials.from_service_account_file(
    key_path, scopes=['https://www.googleapis.com/auth/cloud-platform'])
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print('BigQuery client ready for project:', client.project)

## 2. Query helper and cost discipline

At this scale querying is effectively free: a full scan is ~0.5 GB against BigQuery's
1 TiB/month free tier (cached repeats are free). We still keep two cost-aware habits,
visible in the helper: a **`dry_run`** to preview bytes scanned, and **`maximum_bytes_billed`**
as a guardrail. The six queries are read from the validated files in `sql/` - the same
text used at the BigQuery UI, so there is a single source of truth.

In [ ]:
from pathlib import Path
import pandas as pd

SQL_DIR = Path('sql')

def load_sql(name):
    return (SQL_DIR / name).read_text()

def run_query(sql, max_gb=2.0, preview=True):
    if preview:
        dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
        print(f'dry-run: will scan {dry.total_bytes_processed / 1e9:.3f} GB')
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=int(max_gb * 1e9))
    return client.query(sql, job_config=cfg).to_dataframe()

## 3. Sanity check

Ground-truth anchors before the analysis: total flights, cancellations, and
carrier/airport counts. If these look wrong, a query that runs cleanly could still be
computing the wrong thing.

In [ ]:
run_query(f'''
SELECT
  COUNT(*)                       AS flights,
  COUNTIF(CANCELLED = 1)         AS cancelled,
  COUNT(DISTINCT AIRLINE)        AS carriers,
  COUNT(DISTINCT ORIGIN_AIRPORT) AS origin_airports
FROM {DATASET}.flights
''')

## 4. Q1 - On-time performance by airline

Flights, average departure/arrival delay, and the share of delayed arrivals (>= 15 min,
the DOT threshold) per carrier, joined to the airline name. A carrier reliability ranking.

In [ ]:
q1 = run_query(load_sql('q1_ontime_by_airline.sql'))
q1

In [ ]:
import matplotlib.pyplot as plt

ax = q1.sort_values('pct_delayed').plot.barh(
    x='airline_name', y='pct_delayed', legend=False, figsize=(8, 5), color='steelblue')
ax.set_xlabel('% arrivals delayed (>= 15 min)'); ax.set_ylabel('')
ax.set_title('Q1 - Carrier reliability (lower is better)')
plt.tight_layout(); plt.show()

**Finding.** Ultra-low-cost carriers (Spirit ~30%, Frontier ~27%) are worst; Hawaiian,
Alaska and Delta best (~11-14%). For most carriers `avg_dep_delay > avg_arr_delay` - they
recover time in the air (revisited in Q6).

## 5. Q2 - Delay propagation through the day

Average arrival delay and % delayed by **scheduled departure hour** (derived from the
HHMM field).

In [ ]:
q2 = run_query(load_sql('q2_hourly_propagation.sql'))
q2

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(q2['sched_dep_hour'], q2['avg_arr_delay_min'], marker='o')
ax.axhline(0, color='grey', lw=0.8)
ax.set_xlabel('scheduled departure hour'); ax.set_ylabel('avg arrival delay (min)')
ax.set_title('Q2 - Delays compound through the day')
plt.tight_layout(); plt.show()

**Finding.** Morning flights arrive early; delay rises monotonically to a ~19:00 peak
(+11 min, 27% delayed). Hours 0-4 are very low volume (red-eyes) and noisy - the signal is
the 5-23 rise.

## 6. Q3 - Bottleneck airports

Top origin airports by average departure delay, with a minimum-volume filter, joined to
city/state.

In [ ]:
q3 = run_query(load_sql('q3_bottleneck_airports.sql'))
q3

In [ ]:
ax = q3.sort_values('avg_dep_delay_min').plot.barh(
    x='airport_code', y='avg_dep_delay_min', legend=False, figsize=(8, 6), color='indianred')
ax.set_xlabel('avg departure delay (min)'); ax.set_ylabel('')
ax.set_title('Q3 - Most congested origin airports')
plt.tight_layout(); plt.show()

**Finding.** Chicago O'Hare leads (~14 min), with the NYC-area airports (Newark,
LaGuardia, JFK) and Chicago Midway close behind - the usual congested hubs.

## 7. Q4 - Seasonality

Average arrival delay and cancellation rate by month (October excluded at the source).

In [ ]:
q4 = run_query(load_sql('q4_seasonality.sql'))
q4

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(q4['month'], q4['cancellation_rate_pct'], color='lightcoral', alpha=0.8)
ax1.set_xlabel('month'); ax1.set_ylabel('cancellation rate (%)', color='indianred')
ax2 = ax1.twinx()
ax2.plot(q4['month'], q4['avg_arr_delay_min'], color='navy', marker='o')
ax2.set_ylabel('avg arrival delay (min)', color='navy')
ax1.set_title('Q4 - Winter cancellations, summer delays')
plt.tight_layout(); plt.show()

**Finding.** Cancellations peak in winter (Feb 4.8%, Jan 2.6% - snow); arrival delays
peak in summer (June 9.6 min - thunderstorms + heavy travel). September is the calmest.

## 8. Q5 - Decomposition of delay causes

For delayed flights, the share of total delay minutes attributed to each cause.

In [ ]:
q5 = run_query(load_sql('q5_cause_decomposition.sql'))
q5

In [ ]:
causes = q5.iloc[0].sort_values(ascending=False)
ax = causes.plot.bar(figsize=(7, 4), color='slateblue')
ax.set_ylabel('% of delay minutes'); ax.set_xlabel('')
ax.set_title('Q5 - Late aircraft (propagation) dominates')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

**Finding.** `late_aircraft` (~40%) dominates - this *is* the propagation effect behind
Q2. Airline-controlled (~32%) and air-system (~23%) follow; security is ~0%. Direct
"weather" is only ~5%, but much weather impact hides inside the air-system and late-aircraft
buckets.

## 9. Q6 - Distance vs in-flight recovery

By distance bucket: average `arrival_delay - departure_delay`. Negative means time was made
up in the air.

In [ ]:
q6 = run_query(load_sql('q6_distance_recovery.sql'))
q6

In [ ]:
ax = q6.plot.bar(x='distance_bucket', y='avg_recovery_min', legend=False,
                 figsize=(8, 4), color='seagreen')
ax.axhline(0, color='grey', lw=0.8)
ax.set_ylabel('avg(arrival - departure) delay (min)'); ax.set_xlabel('distance bucket (mi)')
ax.set_title('Q6 - Longer flights recover more time')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

**Finding.** Recovery is negative everywhere and grows with distance (-2.9 -> -9 min):
longer flights have more cruise time to absorb a late departure.

## 10. Data enrichment - BigQuery ML

A binary classifier for **arrival delay >= 15 min**, using only features known *before*
arrival (month, day of week, scheduled hour, airline, origin, destination, distance,
scheduled time). BigQuery ML trains in-database with `CREATE MODEL`. Spark MLlib is the
deferred stage.

> Training scans the full table and takes ~1-2 min; re-run only when needed.

In [ ]:
create_model = f'''
CREATE OR REPLACE MODEL {DATASET}.delay_logreg
OPTIONS(model_type='LOGISTIC_REG', input_label_cols=['is_delayed'], auto_class_weights=TRUE) AS
SELECT
  IF(ARRIVAL_DELAY >= 15, 1, 0) AS is_delayed,
  MONTH, DAY_OF_WEEK,
  MOD(DIV(SAFE_CAST(SCHEDULED_DEPARTURE AS INT64), 100), 24) AS dep_hour,
  AIRLINE, ORIGIN_AIRPORT, DESTINATION_AIRPORT, DISTANCE, SCHEDULED_TIME
FROM {DATASET}.flights
WHERE CANCELLED = 0 AND ARRIVAL_DELAY IS NOT NULL
'''
client.query(create_model).result()
print('model trained')

In [ ]:
run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.delay_logreg`)', preview=False)

**Finding.** The pre-departure features carry real signal (ROC AUC clearly above 0.5):
the same drivers seen descriptively above are also predictive of whether a flight will be
late.

## 11. Conclusions

The six queries tell one coherent story:

- **Propagation is the core mechanism.** Late-arriving aircraft cause ~40% of delay minutes
  (Q5), which is why delays build through the day (Q2) and why on-time rates split carriers (Q1).
- **Geography and season modulate it.** A handful of hubs (Chicago, NYC) inject congestion
  (Q3); winter brings cancellations, summer brings delays (Q4).
- **Physics offers relief.** Longer flights recover late departures in cruise (Q6), already
  visible as `dep_delay > arr_delay` for most carriers (Q1).
- **Prediction.** A pre-departure logistic model (section 10) confirms these features carry
  real signal for "will it be late?".

A **Data Studio** dashboard over these results is the separate deliverable.